In [7]:
%pip install -q pandas


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import importlib

import pandas as pd
import src.constants

importlib.reload(src.constants)
from src.constants import (
    Column,
    INPUT_DIR,
    MODEL_FEATURE_COLUMNS,
    OUTPUT_DIR,
)

In [9]:
electronics_df = pd.read_csv(INPUT_DIR / "electronics_raw.csv")
deterministics_df = pd.read_csv(OUTPUT_DIR / "deterministic_features_results.csv")

output_llama = pd.read_csv(OUTPUT_DIR / "llm" / "llama3_8b_results.csv")
output_qwen = pd.read_csv(OUTPUT_DIR / "llm" / "qwen3_14b_results.csv")

rawAndDeterministic = pd.merge(electronics_df, deterministics_df, "inner", on="id")

final_llama = pd.merge(rawAndDeterministic, output_llama, "inner", on="id")
final_qwen = pd.merge(rawAndDeterministic, output_qwen, "inner", on="id")

def calculate_ratios(df: pd.DataFrame):     
    df[Column.LABEL] = df[Column.LABEL].map({"OR": 0, "CG": 1})

    df[Column.ADJECTIVE_RATIO] = df[Column.ADJECTIVE_COUNT] / df[Column.WORD_COUNT]
    df[Column.VERB_RATIO] = df[Column.VERB_COUNT] / df[Column.WORD_COUNT]
    df[Column.SUPERLATIVE_RATIO] = df[Column.SUPERLATIVE_COUNT] / df[Column.WORD_COUNT]

    df[Column.FIRST_PERSON_PRONOUN_RATIO] = (
        df[Column.FIRST_PERSON_PRONOUN_COUNT] / df[Column.WORD_COUNT]
    )
    df[Column.THIRD_PERSON_PRONOUN_RATIO] = (
        df[Column.THIRD_PERSON_PRONOUN_COUNT] / df[Column.WORD_COUNT]
    )

    claim_count = (
        df[Column.SUBJECTIVE_CLAIM_COUNT]
        + df[Column.OBJECTIVE_CLAIM_COUNT]
    )
    df[Column.SUBJECTIVITY] = (
        df[Column.SUBJECTIVE_CLAIM_COUNT] / claim_count.mask(claim_count == 0)
    ).fillna(0.5)

    normalized_claim_count = claim_count.mask(claim_count == 0)
    df[Column.EXPERIENTIAL_DETAIL] = (
        df[Column.EXPERIENTIAL_DETAIL_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.POSITIVE_AFFECT] = (
        df[Column.POSITIVE_AFFECT_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.NEGATIVE_AFFECT] = (
        df[Column.NEGATIVE_AFFECT_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)
    df[Column.UNCERTAINTY] = (
        df[Column.UNCERTAIN_CLAIM_COUNT] / normalized_claim_count
    ).fillna(0).clip(0, 1)

    text_sentiment_normalized = df[Column.TEXT_SENTIMENT] / 2
    rating_sentiment = (df[Column.RATING] - 3) / 2
    df[Column.INTERNAL_CONSISTENCY] = (
        1 - (text_sentiment_normalized - rating_sentiment).abs() / 2
    ).clip(0, 1)

    print(df.columns)
    print(df.head())

calculate_ratios(final_llama)
calculate_ratios(final_qwen)

final_columns = [Column.ID, Column.LABEL, *MODEL_FEATURE_COLUMNS]
final_llama = final_llama[final_columns].copy()
final_qwen = final_qwen[final_columns].copy()

final_llama.to_csv(OUTPUT_DIR / "final" / "final_llama.csv", index=None)
final_qwen.to_csv(OUTPUT_DIR / "final" / "final_qwen.csv", index=None)

Index(['id', 'category', 'rating', 'label', 'text_', 'word_count',
       'first_person_pronoun_count', 'third_person_pronoun_count',
       'verb_count', 'adjective_count', 'superlative_count', 'readability_ari',
       'extremity', 'subjective_claim_count', 'objective_claim_count',
       'experiential_detail_claim_count', 'positive_affect_claim_count',
       'negative_affect_claim_count', 'uncertain_claim_count',
       'text_sentiment', 'adjective_ratio', 'verb_ratio', 'superlative_ratio',
       'first_person_pronoun_ratio', 'third_person_pronoun_ratio',
       'subjectivity', 'experiential_detail', 'positive_affect',
       'negative_affect', 'uncertainty', 'internal_consistency'],
      dtype='str')
   id       category  rating  label  \
0   0  Electronics_5     2.0      0   
1   1  Electronics_5     5.0      0   
2   2  Electronics_5     5.0      1   
3   3  Electronics_5     5.0      0   
4   4  Electronics_5     5.0      0   

                                               t